<img src="https://www.hl7.org/fhir/assets/images/fhir-logo-www.png" style="float: left; width: 25%; margin-bottom: 0.5em;">
<img src="https://ehealth-graz.at/wp-content/uploads/2023/10/eHealth-Inverted-Color-800x600-1.png" style="float: left; width: 25%; margin-bottom: 0.5em;"> 

Author: Sten Hanke sten.hanke@fh-joanneum.at <br>
Based on: Templates from **Lee Surprenant** lmsurpre@us.ibm.com

# Setting up the server

In [1]:
# save the base url of the FHIR server
#base = 'http://diam.fh-joanneum.at:8021/hapi-fhir-jpaserver/fhir'
base = 'http://hapi.fhir.org/baseR4'

# setup imports
import os
from requests import get
from requests import post
from requests import put
from requests import delete
from requests import head
from IPython.display import IFrame

# a function to print the top x rows and add a newline
def peek(string, line_count=20):
    print(os.linesep.join(string.split(os.linesep)[:line_count]) + '\n')

# Section 1: The FHIR REST API

In [2]:
# HL7 FHIR defines a set of "resources" for exchanging information.
IFrame('https://www.hl7.org/fhir/resourcelist.html#tabs', width=1200, height=330)

In [3]:
# Each resource type supports the same set of interactions, categorized in the spec into "instance-level" and "type-level" interactions.
IFrame('https://www.hl7.org/fhir/http.html#operations', width=1200, height=330)

# Section 2: The FHIR Search

In [5]:
# query for all Patient resources, then print the HTTP status code and the first 25 lines of the response
response = get(base + '/Patient')
print('Response code: ' + str(response.status_code))
peek('Response body: \n' + response.text)
print('Number of entries: ' + str(len(response.json().get('entry'))))

# technically you've now performed your first FHIR "search" (just with no parameters)

Response code: 200
Response body: 
{
  "resourceType": "Bundle",
  "id": "8ca689af-d6b2-41d6-8da3-787e444fe75c",
  "meta": {
    "lastUpdated": "2025-05-23T08:18:41.756+00:00"
  },
  "type": "searchset",
  "link": [ {
    "relation": "self",
    "url": "https://hapi.fhir.org/baseR4/Patient"
  }, {
    "relation": "next",
    "url": "https://hapi.fhir.org/baseR4?_getpages=8ca689af-d6b2-41d6-8da3-787e444fe75c&_getpagesoffset=20&_count=20&_pretty=true&_bundletype=searchset"
  } ],
  "entry": [ {
    "fullUrl": "https://hapi.fhir.org/baseR4/Patient/597083",
    "resource": {
      "resourceType": "Patient",
      "id": "597083",
      "meta": {
        "versionId": "1",
        "lastUpdated": "2020-02-03T12:24:35.175+00:00",
        "source": "#uFRZJnV3kU8uQXri"
      },
      "text": {
        "status": "generated",
        "div": "<div xmlns=\"http://www.w3.org/1999/xhtml\"><div class=\"hapiHeaderText\">Somawathi Piyawathi <b>SIRIYAWATHI </b></div><table class=\"hapiPropertyTable\"><tbod

In [6]:
# note that results are paged and the "link" field in the response Bundle contains links to the previous, current, and next page of results
for link in response.json().get('link'):
    if link.get('relation') == 'next':
        page2 = get(link.get('url'))        
peek('Second page: \n' + page2.text)
print('Number of entries: ' + str(len(response.json().get('entry'))))

Second page: 
{
  "resourceType": "Bundle",
  "id": "8ca689af-d6b2-41d6-8da3-787e444fe75c",
  "meta": {
    "lastUpdated": "2025-05-23T08:18:41.756+00:00"
  },
  "type": "searchset",
  "link": [ {
    "relation": "self",
    "url": "https://hapi.fhir.org/baseR4?_getpages=8ca689af-d6b2-41d6-8da3-787e444fe75c&_getpagesoffset=20&_count=20&_pretty=true&_bundletype=searchset"
  }, {
    "relation": "next",
    "url": "https://hapi.fhir.org/baseR4?_getpages=8ca689af-d6b2-41d6-8da3-787e444fe75c&_getpagesoffset=40&_count=20&_pretty=true&_bundletype=searchset"
  }, {
    "relation": "previous",
    "url": "https://hapi.fhir.org/baseR4?_getpages=8ca689af-d6b2-41d6-8da3-787e444fe75c&_getpagesoffset=0&_count=20&_pretty=true&_bundletype=searchset"
  } ],
  "entry": [ {
    "fullUrl": "https://hapi.fhir.org/baseR4/Patient/597174",
    "resource": {
      "resourceType": "Patient",
      "id": "597174",
      "meta": {
        "versionId": "1",
        "lastUpdated": "2020-02-03T15:55:38.182+00:00",


In [7]:
# we can control the number of resources on each page by passing the _count parameter
response = get(base + '/Patient?_count=1')
peek('Single resource per page: \n' + response.text)
print('Number of entries: ' + str(len(response.json().get('entry'))))

Single resource per page: 
{
  "resourceType": "Bundle",
  "id": "0d11dca6-127d-4fc1-a4dd-f2344815cf78",
  "meta": {
    "lastUpdated": "2025-05-23T08:19:21.593+00:00"
  },
  "type": "searchset",
  "link": [ {
    "relation": "self",
    "url": "https://hapi.fhir.org/baseR4/Patient?_count=1"
  }, {
    "relation": "next",
    "url": "https://hapi.fhir.org/baseR4?_getpages=0d11dca6-127d-4fc1-a4dd-f2344815cf78&_getpagesoffset=1&_count=1&_pretty=true&_bundletype=searchset"
  } ],
  "entry": [ {
    "fullUrl": "https://hapi.fhir.org/baseR4/Patient/597083",
    "resource": {
      "resourceType": "Patient",
      "id": "597083",
      "meta": {
        "versionId": "1",
        "lastUpdated": "2020-02-03T12:24:35.175+00:00",
        "source": "#uFRZJnV3kU8uQXri"
      },
      "text": {
        "status": "generated",
        "div": "<div xmlns=\"http://www.w3.org/1999/xhtml\"><div class=\"hapiHeaderText\">Somawathi Piyawathi <b>SIRIYAWATHI </b></div><table class=\"hapiPropertyTable\"><tbody

In [8]:
# now add some search parameters

# each FHIR resource type has its own set of parameters; find them toward the bottom of the page for that resource type in the specification
# for example, for the Patient resource type, see https://www.hl7.org/fhir/patient.html#search
IFrame('https://www.hl7.org/fhir/patient.html#search', width=1200, height=500)

In [9]:
# for example, lets use the search parameter named "gender"
response = get(base + '/Patient' + '?' + 'gender=male')
print('Response code: ' + str(response.status_code))
peek('Response body: \n' + response.text)

Response code: 200
Response body: 
{
  "resourceType": "Bundle",
  "id": "36af05fd-f49a-46ce-bd57-637e9cf797b3",
  "meta": {
    "lastUpdated": "2025-05-23T08:19:22.651+00:00"
  },
  "type": "searchset",
  "link": [ {
    "relation": "self",
    "url": "https://hapi.fhir.org/baseR4/Patient?gender=male"
  }, {
    "relation": "next",
    "url": "https://hapi.fhir.org/baseR4?_getpages=36af05fd-f49a-46ce-bd57-637e9cf797b3&_getpagesoffset=20&_count=20&_pretty=true&_bundletype=searchset"
  } ],
  "entry": [ {
    "fullUrl": "https://hapi.fhir.org/baseR4/Patient/22",
    "resource": {
      "resourceType": "Patient",
      "id": "22",
      "meta": {
        "versionId": "149",
        "lastUpdated": "2024-04-17T07:06:47.104+00:00",
        "source": "#nascFxu41J3VrEsO"
      },
      "text": {
        "status": "generated",
        "div": "<div xmlns=\"http://www.w3.org/1999/xhtml\"><div class=\"hapiHeaderText\">Sooraj <b>ABRAHAM </b></div><table class=\"hapiPropertyTable\"><tbody><tr><td>I

In [ ]:
# pro tip: combine your search query with _summary=count to explore the data
print('male:   \t' + str(get(base + '/Patient' + '?' + 'gender=male' + '&' + '_summary=count').json().get('total')))
print('female: \t' + str(get(base + '/Patient' + '?' + 'gender=female' + '&' + '_summary=count').json().get('total')))

# use the "missing" modifier to look for resources that do NOT have a value for the target parameter
response = get(base + '/Patient' + '?' + 'gender:missing=true' + '&' + '_summary=count')
print('missing gender: ' + str(response.json().get('total')))

## Section 3: Search parameter types

In [ ]:
# search parameters have types

# gender is considered a "token" search parameter

# Token search
# this parameter type is common for 'coded' values (Code, Coding, and CodeableConcept) and identifiers
# token values consist of a system and a code, although sometimes the system is implicit (like in the case of gender)
# users can search on the system and code (system|code), the code alone (code), system-less codes (|code), or even the system alone (system|)
response = get(base + '/Patient' + '?' + 'gender=http://hl7.org/fhir/administrative-gender|male' + '&_count=1&_elements=gender')
print('male:\n' + response.text)


# there are also Number, Date/DateTime, String, Reference, Quantity, URI, and Composite parameter types

In [ ]:
# String search
response = get(base + '/Patient' + '?' + 'family=Smith' + '&_elements=name')
print('Smiths:')
for entry in response.json().get('entry'):
    resource = entry.get('resource')
    print(resource.get('id'), end=': ')
    print(', '.join(map(lambda n: n.get('family'), resource.get('name'))))

In [ ]:
# wait, "Smitham" !?

# string search performs a case-insensitive "begins-with" search by default!
# use the modifier ":exact" if you want exact matches (and improved performance)
response = get(base + '/Patient' + '?' + 'family:exact=Smith' + '&_elements=name')
print('Smiths:')
for entry in response.json().get('entry'):
    resource = entry.get('resource')
    print(resource.get('id'), end=': ')
    print(', '.join(map(lambda n: n.get('family'), resource.get('name'))))
print()

# string search also has a ":contains" modifier
response = get(base + '/Patient' + '?' + 'family:contains=ster' + '&_elements=name')
print('Ster:')
for entry in response.json().get('entry'):
    resource = entry.get('resource')
    print(resource.get('id'), end=': ')
    print(', '.join(map(lambda n: n.get('family'), resource.get('name'))))

In [ ]:
# Date search
response = get(base + '/Patient' + '?' + 'birthdate=1979' + '&_elements=birthDate')
print('Born in 1979:')
for entry in response.json().get('entry'):
    resource = entry.get('resource')
    print(resource.get('id'), end=': ')
    print(resource.get('birthDate'))

In [ ]:
# date searches support lt(<), le(<=), gt(>), ge(>=), sa(starts after), and eb(ends before) "prefixes"
response = get(base + '/Patient' + '?' + 'birthdate=eb1981' + '&_elements=birthDate')
print('Born before 1979:')
for entry in response.json().get('entry'):
    resource = entry.get('resource')
    print(resource.get('id'), end=': ')
    print(resource.get('birthDate'))

response = get(base + '/Patient' + '?' + 'birthdate=sa1970' + '&_elements=birthDate')
print('\n' + 'Born after 1979:')
for entry in response.json().get('entry'):
    resource = entry.get('resource')
    print(resource.get('id'), end=': ')
    print(resource.get('birthDate'))

# some servers support ap(approximately equal) as well, although the spec lets the server decide exactly what that means...
response = get(base + '/Patient' + '?' + 'birthdate=ap1979' + '&_elements=birthDate')
print('\n' + 'Born "around" 1979:')
for entry in response.json().get('entry'):
    resource = entry.get('resource')
    print(resource.get('id'), end=': ')
    print(resource.get('birthDate'))

## Section 4: Chaining and includes

In [12]:
# Chaining

# where reference parameters get really interesting is when you want to query one resource type based on a property of another resource to which its linked
# for example, here is a search for Type II Diabetes in female patients
response = get(base + '/Condition' + '?' + 'code=http://snomed.info/sct|44054006' + '&' + 'patient:Patient.gender=female' + '&_count=1')
peek('Type II Diabetes in female patients:   \n' + str(response.text), 100)

Type II Diabetes in female patients:   
{
  "resourceType": "Bundle",
  "id": "bc32cf3a-b0ad-4a24-ba65-388555c04c74",
  "meta": {
    "lastUpdated": "2024-03-14T12:08:54.529+00:00"
  },
  "type": "searchset",
  "link": [ {
    "relation": "self",
    "url": "https://hapi.fhir.org/baseR4/Condition?_count=1&code=http%3A%2F%2Fsnomed.info%2Fsct%7C44054006&patient%3APatient.gender=female"
  }, {
    "relation": "next",
    "url": "https://hapi.fhir.org/baseR4?_getpages=bc32cf3a-b0ad-4a24-ba65-388555c04c74&_getpagesoffset=1&_count=1&_pretty=true&_bundletype=searchset"
  } ],
  "entry": [ {
    "fullUrl": "https://hapi.fhir.org/baseR4/Condition/622919",
    "resource": {
      "resourceType": "Condition",
      "id": "622919",
      "meta": {
        "versionId": "1",
        "lastUpdated": "2020-02-18T09:08:21.927+00:00",
        "source": "#8t1bINv1FvZfbmgd",
        "profile": [ "https://www.fhir.philips.com/4.0/StructureDefinition/common/resource/general/condition-v1/ILSCondition" ]
     

In [13]:
# Reverse chaining

# references can be searched the other way around via the "_has" parameter
response = get(base + '/Patient' + '?' + '_has:Condition:patient:code=http://snomed.info/sct|44054006' + '&_count=1')
peek('Patients with Type II Diabetes:   \n' + response.text)

Patients with Type II Diabetes:   
{
  "resourceType": "Bundle",
  "id": "41d49a27-aebf-47db-b449-2c58d7ca8700",
  "meta": {
    "lastUpdated": "2024-03-14T12:09:15.594+00:00"
  },
  "type": "searchset",
  "link": [ {
    "relation": "self",
    "url": "https://hapi.fhir.org/baseR4/Patient?_count=1&_has%3ACondition%3Apatient%3Acode=http%3A%2F%2Fsnomed.info%2Fsct%7C44054006"
  }, {
    "relation": "next",
    "url": "https://hapi.fhir.org/baseR4?_getpages=41d49a27-aebf-47db-b449-2c58d7ca8700&_getpagesoffset=1&_count=1&_pretty=true&_bundletype=searchset"
  } ],
  "entry": [ {
    "fullUrl": "https://hapi.fhir.org/baseR4/Patient/601740",
    "resource": {
      "resourceType": "Patient",
      "id": "601740",



## Section 5: Putting it together

In [14]:
response = get(base + '/Condition' + '?' + 'code=http://snomed.info/sct|44054006' + '&_count=1')
print('Patients with Type II Diabetes:  ' + str(response.json().get('total')))


Patients with Type II Diabetes:  None


In [15]:
# SNOMED concepts for comorbidities of Type II Diabetes
#coronary heart disease (CHD), 53741008
#chronic kidney disease (CKD), 709044004
#atrial fibrillation, 49436004
#stroke, 230690007
#hypertension, 38341003
#heart failure, 84114007
#peripheral vascular disease (PVD), 400047006
#rheumatoid arthritis, 69896004
#Malignant neoplasm, primary (morphologic abnormality), 86049000
#Malignant neoplastic disease (disorder), 363346000
#osteoporosis, 64859006
#depression, 35489007
#asthma, 195967001
#chronic obstructive pulmonary disease (COPD), 13645005
#dementia, 52448006
#severe mental illness (SMI), 391193001
#epilepsy, 84757009
#hypothyroidism, 40930008
#learning disability, 1855002

print('CHD: \t\t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '53741008').json().get('total')))
print('CKD: \t\t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '709044004').json().get('total')))
print('AFib: \t\t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '49436004').json().get('total')))
print('stroke: \t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '230690007').json().get('total')))
print('hypertension: \t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '38341003').json().get('total')))
print('heart failure: \t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '84114007').json().get('total')))
print('PVD: \t\t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '400047006').json().get('total')))
print('arthritis: \t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '69896004').json().get('total')))
print('cancer: \t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '86049000').json().get('total') + get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '363346000').json().get('total')))
print('osteoporosis: \t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '64859006').json().get('total')))
print('depression: \t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '35489007').json().get('total')))
print('asthma: \t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '195967001').json().get('total')))
print('COPD: \t\t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '13645005').json().get('total')))
print('dementia: \t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '52448006').json().get('total')))
print('SMI: \t\t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '391193001').json().get('total')))
print('epilepsy: \t\t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '84757009').json().get('total')))
print('hypothyroidism: \t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '40930008').json().get('total')))
print('learning disability: \t' + str(get(base + '/Condition?_summary=count&code=http://snomed.info/sct|' + '1855002').json().get('total')))

CHD: 			176
CKD: 			10
AFib: 			106
stroke: 		162
hypertension: 		460
heart failure: 		21
PVD: 			5
arthritis: 		544
cancer: 		74
osteoporosis: 		87
depression: 		378
asthma: 		520
COPD: 			107
dementia: 		1
SMI: 			0
epilepsy: 		26
hypothyroidism: 	3
learning disability: 	1
